# BUG 03 — Todos los coeficientes salen con el signo contrario

**Unidad 4.c** · Acompaña a `Clase_05_EleccionBinaria` · Notas: cap. 7

> **Este cuaderno contiene un error deliberado.** No lo corrijas todavía: ejecútalo,
> observa la salida y sigue las tareas del final. Las soluciones están en
> [`SOLUCIONES.md`](SOLUCIONES.md), que conviene no abrir antes de intentarlo.

Estimamos un logit de participación laboral femenina con los datos de Mroz (1987). El
modelo converge sin avisos y entrega errores estándar y valores p perfectamente
normales.

In [1]:
import pandas as pd
import statsmodels.formula.api as smf

datos = pd.read_csv("../../Clase_05_EleccionBinaria/Mroz.csv")

# La columna lfp viene como texto: 'yes' / 'no'. Hay que convertirla a 0/1.
codigos, categorias = pd.factorize(datos["lfp"])
datos["participa"] = codigos

datos[["lfp", "participa"]].head()

,lfp,participa
0,yes,0
1,yes,0
2,yes,0
3,yes,0
4,yes,0


In [2]:
modelo = smf.logit("participa ~ age + k5 + k618 + inc", data=datos).fit(disp=0)

print(f"N = {int(modelo.nobs)}   log-verosimilitud = {modelo.llf:.3f}")
print(f"Convergencia: {modelo.mle_retvals['converged']}\n")

pd.DataFrame(
    {"coef": modelo.params, "ee": modelo.bse, "z": modelo.tvalues, "p": modelo.pvalues}
).round(4)

N = 753   log-verosimilitud = -476.806
Convergencia: True



,coef,ee,z,p
Intercept,-3.9372,0.6088,-6.4668,0.0000
age,0.0660,0.0123,5.3682,0.0000
k5,1.3577,0.1910,7.1096,0.0000
k618,0.1162,0.0657,1.7697,0.0768
inc,0.0181,0.0069,2.6125,0.0090


## Lo único que delata el problema

No hay ningún diagnóstico estadístico que se vea mal. La única señal es que **los signos
contradicen la teoría y la literatura**, y para verlo hay que saber de antemano qué signo
esperar.

Ésta es la razón por la que conviene escribir los signos esperados **antes** de estimar.

In [3]:
esperado = {
    "k5": ("negativo", "hijos menores de 5 años restringen la participación"),
    "k618": ("negativo", "hijos de 6 a 18 años, mismo mecanismo, más débil"),
    "inc": ("negativo", "más ingreso familiar ajeno reduce la necesidad de trabajar"),
    "age": ("negativo", "la participación decae con la edad en esta muestra"),
}

for variable, (signo_teorico, razon) in esperado.items():
    estimado = "positivo" if modelo.params[variable] > 0 else "negativo"
    marca = "  ok" if estimado == signo_teorico else "  <-- CONTRADICE"
    print(f"{variable:6s} estimado {estimado:9s} teoría {signo_teorico:9s}{marca}")
    print(f"       ({razon})")

k5     estimado positivo  teoría negativo   <-- CONTRADICE
       (hijos menores de 5 años restringen la participación)
k618   estimado positivo  teoría negativo   <-- CONTRADICE
       (hijos de 6 a 18 años, mismo mecanismo, más débil)
inc    estimado positivo  teoría negativo   <-- CONTRADICE
       (más ingreso familiar ajeno reduce la necesidad de trabajar)
age    estimado positivo  teoría negativo   <-- CONTRADICE
       (la participación decae con la edad en esta muestra)


El modelo dice que tener hijos pequeños **aumenta** la probabilidad de participar en el
mercado laboral. Eso no es plausible.

## La pregunta que hay que hacerse

¿Qué representa exactamente la variable que estamos modelando?

In [4]:
print(f"Orden de las categorías detectado: {list(categorias)}")
print()
print("pd.factorize asigna los códigos en el orden en que aparecen")
print("las categorías en el archivo, no en orden alfabético ni lógico.")
print()
print(datos[["lfp", "participa"]].drop_duplicates().to_string(index=False))

Orden de las categorías detectado: ['yes', 'no']

pd.factorize asigna los códigos en el orden en que aparecen
las categorías en el archivo, no en orden alfabético ni lógico.

lfp  participa
yes          0
 no          1


## Tareas

1. Localiza la línea culpable. Es una sola.
2. Pregúntale a un asistente de IA qué hace exactamente la función involucrada y de qué
   depende su resultado. **Verifica su respuesta imprimiendo el objeto que devuelve**, no
   confiando en la explicación.
3. Corrige y confirma que los coeficientes se invierten **exactamente**: mismos valores
   absolutos, mismos errores estándar, misma log-verosimilitud, signo opuesto. Explica
   por qué tiene que ser así.
4. ¿Por qué la log-verosimilitud no cambia? ¿Qué dice eso sobre la posibilidad de
   detectar este error con un criterio de bondad de ajuste?

**Regla general:** nunca dejar que el orden del archivo decida la codificación de una
variable categórica cuyo signo se va a interpretar. El mismo cuidado aplica a
`astype('category').cat.codes` y a `LabelEncoder`.

---
Parte del curso de **Econometría I**, Facultad de Ciencias, UNAM.
Ver el [README de la carpeta](README.md) para las otras actividades de depuración.

In [5]:
# Tu corrección aquí.
#
# Pista: codifica explícitamente, nombrando la categoría que vale 1.